# ImmoScore — estimation du prix d'un bien

**Activité 0 · le travail pré-MLOps**

L'agence ImmoScore veut afficher une estimation de prix sur son site. Première étape : vérifier
qu'un modèle simple peut prédire le prix de vente à partir des caractéristiques d'une maison.

Jeu de données : [House Prices — Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)
(ventes réelles à Ames, Iowa, 2006-2010).

**Les huit étapes de l'énoncé**

1. Charger les bibliothèques
2. Charger le fichier dans un dataframe
3. Ne garder que les variables numériques, et supprimer les lignes incomplètes
4. Séparer les variables explicatives (X) de la cible (y : `SalePrice`)
5. Découper en entraînement et test, avec stratification par binning de la cible
6. Entraîner une régression linéaire
7. Mesurer et afficher RMSE et R², sur l'entraînement **et** sur le test
8. Sauvegarder le modèle dans `model.pkl`

À la fin, reportez au tableau : votre **RMSE de test**, votre **R² de test**, et vos **versions**
de `scikit-learn` et de `pandas`.

## 1 · Charger les bibliothèques

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 2 · Charger le fichier dans un dataframe

Le fichier s'appelle `train.csv` sur Kaggle. Si vous travaillez dans un notebook Kaggle, il est
déjà monté ; sinon, déposez-le à côté de ce notebook.

In [ ]:
CHEMINS = [
    "train.csv",
    "data/train.csv",
    "/kaggle/input/house-prices-advanced-regression-techniques/train.csv",
]
chemin = next((c for c in CHEMINS if Path(c).exists()), None)
if chemin is None:
    raise FileNotFoundError(f"train.csv introuvable. Cherche dans : {CHEMINS}")

df = pd.read_csv(chemin)
print(f"Fichier      : {chemin}")
print(f"Dimensions   : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

## 3 · Ne garder que les variables numériques, et supprimer les lignes incomplètes

Une régression linéaire ne sait pas traiter directement les variables textuelles. On écarte aussi
la colonne `Id`, qui est un identifiant et non une caractéristique du bien.

In [ ]:
avant = df.shape

num = df.select_dtypes(include=[np.number]).drop(columns=["Id"], errors="ignore")
num = num.dropna()

print(f"Avant  : {avant[0]} lignes, {avant[1]} colonnes")
print(f"Apres  : {num.shape[0]} lignes, {num.shape[1]} colonnes")
print(f"Perdu  : {avant[0] - num.shape[0]} lignes "
      f"({100 * (avant[0] - num.shape[0]) / avant[0]:.1f} %)")

## 4 · Séparer les variables explicatives de la cible

In [ ]:
X = num.drop(columns=["SalePrice"])
y = num["SalePrice"]

print(f"X : {X.shape[0]} lignes, {X.shape[1]} variables")
print(f"y : {y.shape[0]} valeurs, de {y.min():,.0f} $ a {y.max():,.0f} $")

## 5 · Découper en entraînement et test, avec stratification

Les prix sont très inégalement répartis : quelques biens très chers pèsent lourd. Un découpage
purement aléatoire risquerait de concentrer ces biens d'un seul côté. On découpe donc la cible en
tranches, et on demande que chaque tranche soit représentée dans les mêmes proportions des deux
côtés.

In [ ]:
tranches = pd.qcut(y, q=5, labels=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=tranches
)

print(f"Entrainement : {X_train.shape[0]} biens")
print(f"Test         : {X_test.shape[0]} biens")

## 6 · Entraîner une régression linéaire

In [ ]:
modele = LinearRegression()
modele.fit(X_train, y_train)

print(f"Modele entraine sur {X_train.shape[1]} variables.")

## 7 · Mesurer et afficher RMSE et R²

On mesure sur l'entraînement **et** sur le test. L'écart entre les deux indique si le modèle a
appris des régularités du marché ou des particularités de son échantillon.

In [ ]:
def mesurer(modele, X, y):
    prediction = modele.predict(X)
    rmse = np.sqrt(mean_squared_error(y, prediction))
    return rmse, r2_score(y, prediction)

rmse_train, r2_train = mesurer(modele, X_train, y_train)
rmse_test,  r2_test  = mesurer(modele, X_test,  y_test)

print(f"Entrainement   RMSE = {rmse_train:>10,.0f} $    R2 = {r2_train:.3f}")
print(f"Test           RMSE = {rmse_test:>10,.0f} $    R2 = {r2_test:.3f}")
print(f"Ecart RMSE     {rmse_test - rmse_train:>10,.0f} $")

## 8 · Sauvegarder le modèle

In [ ]:
with open("model.pkl", "wb") as f:
    pickle.dump(modele, f)

print(f"Modele enregistre : model.pkl ({Path('model.pkl').stat().st_size / 1024:.1f} Ko)")

---

## À reporter au tableau

Inscrivez ces quatre valeurs au tableau, à côté de celles de vos camarades.

In [ ]:
print(f"RMSE de test  : {rmse_test:,.0f} $")
print(f"R2 de test    : {r2_test:.3f}")
print(f"scikit-learn  : {sklearn.__version__}")
print(f"pandas        : {pd.__version__}")